Import Libraries

In [1]:
import pandas as pd
import numpy as np

import mlflow
import mlflow.sklearn

import optuna

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score
)

from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score
)

d:\Important Files\bank_marketing_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load Machine Learning Dataset

In [2]:
import pandas as pd

# Load processed dataset
df = pd.read_csv("../src/data/processed/bank_ml_dataset.csv")

print("Dataset Loaded Successfully")
print(df.shape)

df.head()

Dataset Loaded Successfully
(41176, 54)


,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success,y
0,56,261,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,False,False,False,True,False,False,False,True,False,0
1,57,149,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,False,False,False,True,False,False,False,True,False,0
2,37,226,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,False,False,False,True,False,False,False,True,False,0
3,40,151,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,False,False,False,True,False,False,False,True,False,0
4,56,307,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,False,False,False,True,False,False,False,True,False,0


Split Features and Target

In [3]:
X = df.drop("y", axis=1)
y = df["y"]

print("Features Shape :", X.shape)
print("Target Shape :", y.shape)

Features Shape : (41176, 53)
Target Shape : (41176,)


Train Test Split

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Shape :", X_train.shape)
print("Test Shape :", X_test.shape)

Train Shape : (32940, 53)
Test Shape : (8236, 53)


Feature Scaling

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Feature Scaling Completed")

Feature Scaling Completed


Cross Validation (Logistic Regression)

In [6]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import StratifiedKFold, cross_val_score

# log_model = LogisticRegression(max_iter=1000, random_state=42)

# cv = StratifiedKFold(
#     n_splits=5,
#     shuffle=True,
#     random_state=42
# )

# cv_scores = cross_val_score(
#     log_model,
#     X_train,
#     y_train,
#     cv=cv,
#     scoring="accuracy"
# )

# print("Cross Validation Scores")
# print(cv_scores)

# print("\nAverage Accuracy:")
# print(cv_scores.mean())

# print("\nStandard Deviation:")
# print(cv_scores.std())

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

log_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    log_model,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy"
)
print("CROSS VALIDATION RESULTS")

for i, score in enumerate(cv_scores, start=1):
    print(f"Fold {i}: {score*100:.2f}%")

print("\nAverage Accuracy : {:.2f}%".format(cv_scores.mean()*100))
print("Standard Deviation: {:.2f}%".format(cv_scores.std()*100))

CROSS VALIDATION RESULTS
Fold 1: 91.14%
Fold 2: 91.07%
Fold 3: 90.88%
Fold 4: 91.11%
Fold 5: 91.35%

Average Accuracy : 91.11%
Standard Deviation: 0.15%


MLflow Experiment

In [8]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment("Bank Marketing Prediction")

<Experiment: artifact_location='file:///d:/Important Files/bank_marketing_project/notebooks/mlruns/1', creation_time=1782921317491, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1782921317491, lifecycle_stage='active', name='Bank Marketing Prediction', tags={}, trace_location=None, workspace='default'>

Train Model + Log in MLflow

In [9]:
with mlflow.start_run():

    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)

    prediction = model.predict(X_test)

    accuracy = accuracy_score(y_test, prediction) * 100
    roc = roc_auc_score(y_test, prediction)

    mlflow.log_param("Model", "Logistic Regression")
    mlflow.log_metric("Accuracy", accuracy)
    mlflow.log_metric("ROC AUC", roc)

    mlflow.sklearn.log_model(model, "LogisticRegression")

    print("Run Logged Successfully")

2026/07/01 22:22:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run Logged Successfully


Open MLflow UI

In [10]:
print("OPEN MLFLOW UI")

print("Step 1: Open a new terminal.")
print("Step 2: Activate virtual environment.")
print("Step 3: Run command:")
print("mlflow ui")
print("\nThen open:")
print("http://127.0.0.1:5000")

OPEN MLFLOW UI
Step 1: Open a new terminal.
Step 2: Activate virtual environment.
Step 3: Run command:
mlflow ui

Then open:
http://127.0.0.1:5000


Hyperparameter Tuning

In [11]:
def objective(trial):

    C = trial.suggest_float("C", 0.01, 10.0, log=True)

    model = LogisticRegression(
        C=C,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)

    prediction = model.predict(X_test)

    accuracy = accuracy_score(y_test, prediction) * 100

    return accuracy

Run Optuna Study

In [12]:
study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=20)

print("Best Accuracy: {:.2f}%".format(study.best_value))
print("Best Parameters:")
print(study.best_params)

[I 2026-07-01 22:22:21,237] A new study created in memory with name: no-name-9e52ac94-ab01-483e-886b-da7b4a27727b
[I 2026-07-01 22:22:21,520] Trial 0 finished with value: 90.93006313744536 and parameters: {'C': 0.8031534335347148}. Best is trial 0 with value: 90.93006313744536.
[I 2026-07-01 22:22:21,738] Trial 1 finished with value: 90.90577950461389 and parameters: {'C': 9.765273462036546}. Best is trial 0 with value: 90.93006313744536.
[I 2026-07-01 22:22:21,968] Trial 2 finished with value: 90.99077221952405 and parameters: {'C': 0.4011693099726589}. Best is trial 2 with value: 90.99077221952405.
[I 2026-07-01 22:22:22,114] Trial 3 finished with value: 91.00291403593978 and parameters: {'C': 0.09476062659760913}. Best is trial 3 with value: 91.00291403593978.
[I 2026-07-01 22:22:22,218] Trial 4 finished with value: 91.00291403593978 and parameters: {'C': 0.015630254632608188}. Best is trial 3 with value: 91.00291403593978.
[I 2026-07-01 22:22:22,341] Trial 5 finished with value: 91

Best Accuracy: 91.04%
Best Parameters:
{'C': 0.03304197743666144}


Train Final Tuned Model

In [13]:
best_model = LogisticRegression(
    C=study.best_params["C"],
    max_iter=1000,
    random_state=42
)

best_model.fit(X_train, y_train)

prediction = best_model.predict(X_test)

accuracy = accuracy_score(y_test, prediction) * 100

roc = roc_auc_score(y_test, prediction) * 100

print("TUNED MODEL RESULTS")

print(f"Accuracy : {accuracy:.2f}%")
print(f"ROC AUC  : {roc:.2f}%")

TUNED MODEL RESULTS
Accuracy : 91.04%
ROC AUC  : 69.93%


Save Tuned Model

In [14]:
import joblib

joblib.dump(
    best_model,
    "../src/models/logistic_regression_tuned.pkl"
)

print("Tuned Model Saved Successfully.")

Tuned Model Saved Successfully.


Best Model Save

In [15]:
import joblib

joblib.dump(model, "../src/models/best_model.pkl")

print("Best Model Saved Successfully.")

Best Model Saved Successfully.


Save Best Parameters

In [16]:
best_params = pd.DataFrame([study.best_params])

best_params.to_csv(
    "../reports/best_parameters.csv",
    index=False
)

print("Best Parameters Saved.")
best_params

Best Parameters Saved.


,C
0,0.033042


Save Optuna Results

In [17]:
results = pd.DataFrame(study.trials_dataframe())

results.to_csv(
    "../reports/optuna_trials.csv",
    index=False
)

print("Optuna Trial Results Saved.")

Optuna Trial Results Saved.


Final Summary

In [18]:
print("HYPERPARAMETER TUNING COMPLETED")

print(f"Best Accuracy : {study.best_value:.2f}%")
print()
print("Best Parameters:")
print(study.best_params)
print()

print("Files Saved Successfully")
print("best_model.pkl")
print("best_parameters.csv")
print("optuna_trials.csv")
print("=" * 60)

HYPERPARAMETER TUNING COMPLETED
Best Accuracy : 91.04%

Best Parameters:
{'C': 0.03304197743666144}

Files Saved Successfully
best_model.pkl
best_parameters.csv
optuna_trials.csv
